# Image-to-Video Generation with Diffusers/Transformers

This notebook demonstrates how to generate videos from a single image using diffusion models.

## What This Notebook Does

1. Takes your input image
2. Uses it as a starting point for video generation
3. Extends the image into a short animated video
4. Saves the output as a video file

## Requirements

- Python 3.8+
- PyTorch
- diffusers library
- transformers library
- Pillow (for image processing)
- ffmpeg (for video encoding)

## Hardware Requirements

- **Minimum**: 4GB RAM (smaller models only)
- **Recommended**: 8GB+ RAM or 4GB+ VRAM
- **Optimal**: 16GB+ RAM or 8GB+ VRAM

## How It Works (Simple Explanation)

1. **Image Encoder**: Converts your image into numbers (embeddings)
2. **Noise Scheduler**: Starts with a slight variation of your image
3. **U-Net Model**: Repeatedly refines the image, adding motion and transitions
4. **Frame Generation**: Creates intermediate frames between the starting state
5. **Video Assembly**: Combines frames into a video file

## Supported Image Formats

- JPG, PNG, GIF, BMP, TIFF
- Recommended: JPG or PNG with 1920x1080 or smaller

## Getting Help

If you see errors:
1. Check your image isn't be too large (try 512x512 or smaller)
2. Check you have enough RAM/VRAM
3. Try a smaller model
4. Check the documentation in `docs/` folder

In [ ]:
# Step 1: Import required libraries
import torch
import numpy as np
from PIL import Image
import os
import sys

print("Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Load and prepare the input image

def load_image(image_path, max_size=512):
    """Load and resize an image for video generation""
    
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    # Load image
    image = Image.open(image_path).convert('RGB')
    
    # Resize if needed (for memory efficiency)
    width, height = image.size
    if max(width, height) > max_size:
        ratio = max_size / max(width, height)
        new_width = int(width * ratio)
        new_height = int(height * ratio)
        image = image.resize((new_width, new_height))
        print(f"Resized from {width}x{height} to {new_width}x{new_height}")
    
    print(f"Image loaded: {image.size[0]}x{image.size[1]} pixels")
    return image

# Replace this with your your image path
image_path = "input_image.jpg"  # Change this to your your image file

if os.path.exists(image_path):
    image = load_image(image_path)
    print(f"Image ready for video generation")
else:
    print(f"Image not found: {image_path}")
    print("\nTo use your own image:")
    print("1. Put your image file in this folder")
    print("2. Change 'image_path' above to your file name")
    print("3. Supported formats: JPG, PNG, GIF, BMP, TIFF")
    print("\nFor demo, we will use a placeholder image...")

In [ ]:
# Step 3: Load the model

def load_model(model_name="stabilityai/stable-diffusion-xl", device="auto"):
    """Load a text-to-video model for image-to-video""
    
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    
    print(f"Loading model on {device}...")
    
    try:
        pipe = pipeline(
            "text-to-image",  # We'll use text-to-image for image-to-video
            model=model_name,
            device=0 if device == "cuda" else -1
        )
        print(f"Model loaded successfully!")
        return pipe, device
    except Exception as e:
        print(f"Error loading model: {e}")
        raise

# Load model
model_name = "stabilityai/stable-diffusion-xl"
pipe, device = load_model(model_name)

In [ ]:
# Step 4: Generate video from image

def generate_video_from_image(pipe, image, prompt, num_frames=16, device="cpu"):
    """Generate a video extending from an input image""
    
    print(f"\nGenerating video from image...")
    print(f"Prompt: {prompt}")
    print(f"Frames: {num_frames}")
    print(f"Device: {device}")
    print("This may take a few minutes...")
    
    frames = []
    
    try:
        # Generate each frame
        for i in range(num_frames):
            # Create varied prompt for each frame
            frame_prompt = f"{prompt}, frame {i+1}/{num_frames}"
            
            # Generate frame (using image as starting point)
            frame = pipe(
                frame_prompt,
                image=image,
                num_inference_steps=30,
            )
            
            frames.append(frame)
            print(f"Frame {i+1}/{num_frames} generated")
        
        print(f"\nGenerated {len(frames)} frames!")
        return frames
        
    except Exception as e:
        print(f"\nError during generation: {e}")
        raise

# Your text prompt for the video
prompt = "A peaceful animated scene with gentle motion"

# Generate video
frames = generate_video_from_image(
    pipe,
    image,
    prompt,
    num_frames=8,  # Start with fewer frames for testing
    device=device
)

In [ ]:
# Step 5: Save the generated video

import cv2

def save_video(frames, prompt, output_dir="output"):
    """Save generated frames as a video file""
    
    os.makedirs(output_dir, exist_ok=True)
    
    safe_prompt = ''.join(c if c.isalnum() or c in ' -_' else '_' for c in prompt[:50])
    output_path = os.path.join(output_dir, f"video_from_image_{safe_prompt}.mp4")
    
    if not frames:
        print("No frames to save!")
        return None
    
    # Get frame dimensions
    first_frame = frames[0]
    if hasattr(first_frame, 'size'):
        height, width = first_frame.size[1], first_frame.size[0]
    else:
        height, width = first_frame.shape[:2]
    
    # Create video writer
    fourcc = cv2.VideoWriter_fourcc('mp4v')
    video = cv2.VideoWriter(output_path, fourcc, 4.0, (width, height))
    
    # Write each frame
    for frame in frames:
        if hasattr(frame, 'convert'):
            frame = frame.convert('RGB')
        
        if hasattr(frame, 'numpy'):
            frame_np = frame.numpy()
        else:
            frame_np = np.array(frame)
        
        frame_bgr = cv2.cvtColor(frame_np, cv2.COLOR_RGB2BGR)
        video.write(frame_bgr)
    
    video.release()
    print(f"\nVideo saved: {output_path}")
    return output_path

# Save the generated video
video_path = save_video(frames, prompt)
print(f"\nYour video is ready at: {video_path}")

## Next Steps

1. **Watch your video**: Open the saved file in your video player
2. **Try different images**: Replace `image_path` with your your image
3. **Try different prompts**: Change the `prompt` variable
4. **Adjust quality**: Change `num_frames` for different lengths

## Troubleshooting

### Out Memory Error
Reduce `num_frames` to 4-6 and use smaller images (256x256 or smaller)

### Slow Generation
Use fewer frames, lower resolution, or enable GPU if available

### Image Not Found
1. Put your image in this folder
2. Update `image_path` variable
3. Check file extension (JPG, PNG, GIF, BMP, TIFF supported)